# 🍳 SmolLM2-360M Recipe & Nutrition Fine-Tuning Pipeline

**Complete pipeline:** Data download → Clean → Format → Fine-tune → Quantize → Test

**Target:** SmolLM2-360M → LoRA fine-tune → GGUF Q4 quantization → Deploy on 2GB board

**GPU:** T4 (Colab free tier) — ~2-4 hours for 100K examples

---

## Cell 1: Install Dependencies

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
import random
import re

# ============================
# Text Cleaning
# ============================
def clean_text(text):
    """Clean and normalize text."""
    if not text:
        return ""

    text = str(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Keep useful punctuation only
    text = re.sub(r"[^\w\s.,;:!?\'\"()/\-]", "", text)

    return text

## Cell 2: Mount Google Drive (for saving model)
Optional but recommended — Colab sessions can disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project folder
import os
PROJECT_DIR = '/content/drive/MyDrive/smollm2-recipe'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Project dir: {PROJECT_DIR}")

Mounted at /content/drive
Project dir: /content/drive/MyDrive/smollm2-recipe


## Cell 3: Download Raw Data
- **RecipeNLG** — ~2M recipes from HuggingFace
- **USDA FoodData Central** — nutrition data from HuggingFace

In [ ]:
from datasets import load_dataset
import json
import random

# ============================
# 1. RECIPE DATASET
# ============================
print("Loading RecipeNLG 50k...")
recipe_dataset = load_dataset("EmTpro01/recipe-nlg-50k", split="train")
print(f"Loaded {len(recipe_dataset)} recipes")

# ============================
# 2. NUTRITION DATA (HF ONLY - streaming)
# ============================
print("Loading Open Food Facts from Hugging Face (streaming)...")
food_stream = load_dataset("openfoodfacts/product-database", split="food", streaming=True)

food_data = []
nutrition_items = []
count = 0

for ex in food_stream:
    if count >= 60000:
        break
    count += 1

    name = ex.get("product_name", "")
    calories = ex.get("energy-kcal_100g")
    protein = ex.get("proteins_100g")
    carbs = ex.get("carbohydrates_100g")
    fat = ex.get("fat_100g")
    fiber = ex.get("fiber_100g")
    sugar = ex.get("sugars_100g")

    if not name or calories is None:
        continue

    # Build nutrition_items for the detailed pipeline (Cells 9-15)
    try:
        nutrition_items.append({
            "name": name.strip(),
            "serving_size": "100g",
            "calories": round(float(calories), 1),
            "protein": round(float(protein), 1) if protein else 0.0,
            "carbs": round(float(carbs), 1) if carbs else 0.0,
            "fat": round(float(fat), 1) if fat else 0.0,
            "fiber": round(float(fiber), 1) if fiber else None,
            "sugar": round(float(sugar), 1) if sugar else None,
        })
    except (ValueError, TypeError):
        continue

    # Build quick chat dataset
    food_data.append({
        "messages": [
            {"role": "system", "content": "You are a precise nutrition assistant."},
            {"role": "user", "content": f"How many calories are in {name}?"},
            {"role": "assistant", "content": f"{name} has {calories} kcal per 100g."}
        ]
    })

# ============================
# 3. SAVE QUICK DATASET
# ============================
random.shuffle(food_data)

with open("food_chat_dataset.jsonl", "w") as f:
    for item in food_data:
        json.dump(item, f)
        f.write("\n")

print(f"Nutrition items built: {len(nutrition_items)}")
print(f"Quick chat samples: {len(food_data)}")
print("Saved → food_chat_dataset.jsonl")

Loading RecipeNLG 50k...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/408 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/13.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Loaded 50000 recipes
Loading Open Food Facts from Hugging Face (streaming)...


README.md:   0%|          | 0.00/3.19k [00:00<?, ?B/s]

Nutrition items built: 0
Quick chat samples: 0
Saved → food_chat_dataset.jsonl


## Cell 5: (Optional) Add Manual Nutrition Items
For common foods the USDA API might miss or if you want to ensure coverage.

In [ ]:
# Add common foods manually if USDA didn't cover them well
# These are approximate values per 100g
manual_foods = [
    {"name": "Banana", "serving_size": "100g", "calories": 89, "protein": 1.1, "carbs": 22.8, "fat": 0.3, "fiber": 2.6, "sugar": 12.2},
    {"name": "White Rice (cooked)", "serving_size": "100g", "calories": 130, "protein": 2.7, "carbs": 28.2, "fat": 0.3, "fiber": 0.4, "sugar": 0.1},
    {"name": "Brown Rice (cooked)", "serving_size": "100g", "calories": 123, "protein": 2.7, "carbs": 25.6, "fat": 1.0, "fiber": 1.8, "sugar": 0.4},
    {"name": "Chicken Breast (cooked)", "serving_size": "100g", "calories": 165, "protein": 31.0, "carbs": 0.0, "fat": 3.6, "fiber": 0.0, "sugar": 0.0},
    {"name": "Salmon (cooked)", "serving_size": "100g", "calories": 208, "protein": 20.4, "carbs": 0.0, "fat": 13.4, "fiber": 0.0, "sugar": 0.0},
    {"name": "Egg (whole, boiled)", "serving_size": "100g", "calories": 155, "protein": 12.6, "carbs": 1.1, "fat": 10.6, "fiber": 0.0, "sugar": 1.1},
    {"name": "Avocado", "serving_size": "100g", "calories": 160, "protein": 2.0, "carbs": 8.5, "fat": 14.7, "fiber": 6.7, "sugar": 0.7},
    {"name": "Broccoli (cooked)", "serving_size": "100g", "calories": 35, "protein": 2.4, "carbs": 7.2, "fat": 0.4, "fiber": 3.3, "sugar": 1.4},
    {"name": "Sweet Potato (baked)", "serving_size": "100g", "calories": 90, "protein": 2.0, "carbs": 20.7, "fat": 0.1, "fiber": 3.3, "sugar": 6.5},
    {"name": "Oats (dry)", "serving_size": "100g", "calories": 389, "protein": 16.9, "carbs": 66.3, "fat": 6.9, "fiber": 10.6, "sugar": 0.0},
    {"name": "Greek Yogurt (plain)", "serving_size": "100g", "calories": 59, "protein": 10.0, "carbs": 3.6, "fat": 0.7, "fiber": 0.0, "sugar": 3.2},
    {"name": "Almonds", "serving_size": "100g", "calories": 579, "protein": 21.2, "carbs": 21.6, "fat": 49.9, "fiber": 12.5, "sugar": 4.4},
    {"name": "Tofu (firm)", "serving_size": "100g", "calories": 144, "protein": 17.3, "carbs": 2.8, "fat": 8.7, "fiber": 2.3, "sugar": 0.7},
    {"name": "Lentils (cooked)", "serving_size": "100g", "calories": 116, "protein": 9.0, "carbs": 20.1, "fat": 0.4, "fiber": 7.9, "sugar": 1.8},
    {"name": "Peanut Butter", "serving_size": "100g", "calories": 588, "protein": 25.1, "carbs": 20.0, "fat": 50.4, "fiber": 6.0, "sugar": 9.2},
    {"name": "Whole Wheat Bread", "serving_size": "100g", "calories": 247, "protein": 13.0, "carbs": 41.3, "fat": 3.4, "fiber": 6.8, "sugar": 5.6},
    {"name": "Apple", "serving_size": "100g", "calories": 52, "protein": 0.3, "carbs": 13.8, "fat": 0.2, "fiber": 2.4, "sugar": 10.4},
    {"name": "Mango", "serving_size": "100g", "calories": 60, "protein": 0.8, "carbs": 15.0, "fat": 0.4, "fiber": 1.6, "sugar": 13.7},
    {"name": "Spinach (raw)", "serving_size": "100g", "calories": 23, "protein": 2.9, "carbs": 3.6, "fat": 0.4, "fiber": 2.2, "sugar": 0.4},
    {"name": "Milk (whole)", "serving_size": "100ml", "calories": 61, "protein": 3.2, "carbs": 4.8, "fat": 3.3, "fiber": 0.0, "sugar": 5.1},
]

# Merge — avoid duplicates by name
existing_names = {item['name'].lower() for item in nutrition_items}
for mf in manual_foods:
    if mf['name'].lower() not in existing_names:
        nutrition_items.append(mf)

print(f"Total nutrition items after manual additions: {len(nutrition_items)}")

Total nutrition items after manual additions: 20


## Cell 6: Define All Prompt Templates and Formatters

In [ ]:
import random
import re


# ============================
# Recipe Prompts
# ============================
recipe_prompts = [
    "Give me the recipe for {dish}",
    "How do I make {dish}?",
    "Recipe for {dish}",
    "How do you cook {dish}?",
    "I want to make {dish}, what do I need?",
    "Show me how to prepare {dish}",
    "What is the recipe for {dish}?",
    "Can you give me a recipe for {dish}?",
    "Teach me how to make {dish}",
    "{dish} recipe",
    "Easy recipe for {dish}",
    "Simple way to cook {dish}",
]


# ============================
# Nutrition Prompts
# ============================
nutrition_prompts = [
    "What is the nutritional info for {food}?",
    "Nutrition facts for {food}",
    "How healthy is {food}?",
    "What nutrients are in {food}?",
    "Nutritional value of {food}",
    "Tell me the nutrition in {food}",
    "{food} nutrition",
    "Is {food} healthy?",
]


# ============================
# Calorie Prompts
# ============================
calorie_prompts = [
    "How many calories are in {food}?",
    "Calories in {food}",
    "What is the calorie count of {food}?",
    "Tell me the calories in {food}",
    "{food} calories",
    "Does {food} have a lot of calories?",
]


# ============================
# Protein Prompts
# ============================
protein_prompts = [
    "How much protein is in {food}?",
    "Protein in {food}",
    "How many grams of protein are in {food}?",
    "What is the protein content of {food}?",
    "Tell me the protein in {food}",
    "{food} protein",
    "Is {food} high in protein?",
]


# ============================
# Carb Prompts
# ============================
carb_prompts = [
    "How many carbs are in {food}?",
    "Carbs in {food}",
    "How many carbohydrates are in {food}?",
    "What is the carb content of {food}?",
    "Tell me the carbs in {food}",
    "{food} carbs",
]


# ============================
# Fat Prompts
# ============================
fat_prompts = [
    "How much fat is in {food}?",
    "Fat in {food}",
    "How many grams of fat are in {food}?",
    "What is the fat content of {food}?",
    "Tell me the fat in {food}",
    "{food} fat",
]


# ============================
# Fiber Prompts
# ============================
fiber_prompts = [
    "How much fiber is in {food}?",
    "Fiber in {food}",
    "What is the fiber content of {food}?",
    "Tell me the fiber in {food}",
    "{food} fiber",
]


# ============================
# Sugar Prompts
# ============================
sugar_prompts = [
    "How much sugar is in {food}?",
    "Sugar in {food}",
    "How many grams of sugar are in {food}?",
    "Is {food} high in sugar?",
    "Tell me the sugar content of {food}",
    "{food} sugar",
]


# ============================
# Macro Prompts
# ============================
macro_prompts = [
    "What are the macros for {food}?",
    "Macros in {food}",
    "Give me the macros for {food}",
    "Protein carbs and fat in {food}?",
    "What are the macronutrients in {food}?",
    "Tell me the macros of {food}",
    "Break down the macros in {food}",
    "{food} macros",
]


# ============================
# Comparison Prompts
# ============================
compare_prompts = [
    "What has more protein, {food1} or {food2}?",
    "Which has fewer calories, {food1} or {food2}?",
    "Compare {food1} and {food2} nutritionally",
    "Which is healthier, {food1} or {food2}?",
    "{food1} vs {food2} nutrition",
    "Which is better for weight loss, {food1} or {food2}?",
]


# ============================
# Conversational Prompts
# ============================
conversation_prompts = [
    "Is {food} good for weight loss?",
    "Can I eat {food} every day?",
    "Is {food} healthy for gym?",
    "Should I eat {food} before a workout?",
    "What are healthy foods high in protein?",
    "What should I eat after a workout?",
    "What are good low calorie foods?",
    "What foods help build muscle?",
]


print("All prompt templates loaded successfully.")

All prompt templates loaded successfully.


## Cell 7: Define All Formatting Functions

In [ ]:
import random
import json

# ============================
# Validation
# ============================
def is_valid_recipe(entry):
    ingredients = entry.get("ingredients", entry.get("ner", []))
    directions = entry.get("directions", entry.get("steps", []))
    title = entry.get("title", "")

    return (
        title
        and len(title) > 3
        and len(title) < 100
        and len(ingredients) >= 2
        and len(directions) >= 2
        and "test" not in title.lower()
    )


def is_valid_nutrition(item):
    try:
        return (
            item.get("name")
            and item.get("calories") is not None
            and item.get("protein") is not None
            and item.get("carbs") is not None
            and item.get("fat") is not None
            and 0 < float(item["calories"]) < 5000
        )
    except:
        return False


# ============================
# Recipe Formatting
# ============================
def format_recipe(raw):
    title = clean_text(raw.get("title", ""))

    ingredients = raw.get("ingredients", raw.get("ner", []))
    directions = raw.get("directions", raw.get("steps", []))

    # Handle <extra_id_99> separator used in EmTpro01/recipe-nlg-50k
    if isinstance(ingredients, str):
        if "<extra_id_99>" in ingredients:
            ingredients = ingredients.split("<extra_id_99>")
        else:
            try:
                ingredients = json.loads(ingredients)
            except:
                ingredients = ingredients.split(",")

    if isinstance(directions, str):
        if "<extra_id_99>" in directions:
            directions = directions.split("<extra_id_99>")
        else:
            try:
                directions = json.loads(directions)
            except:
                directions = directions.split(".")

    ingredients = [
        clean_text(i)
        for i in ingredients
        if len(str(i).strip()) > 2
    ]

    steps = [
        clean_text(s)
        for s in directions
        if len(str(s).strip()) > 5
    ]

    # Keep recipes SHORT for tiny models
    ingredients = ingredients[:10]
    steps = steps[:6]

    if not ingredients or not steps:
        return None

    ing_text = "\n".join(f"- {i}" for i in ingredients)

    step_text = "\n".join(
        f"{idx+1}. {step}"
        for idx, step in enumerate(steps)
    )

    prompt = random.choice(recipe_prompts).format(dish=title)

    return {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful recipe assistant. "
                    "Give concise and accurate recipes."
                )
            },
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": (
                    f"Recipe: {title}\n\n"
                    f"Ingredients:\n{ing_text}\n\n"
                    f"Steps:\n{step_text}"
                )
            }
        ]
    }


# ============================
# Full Nutrition Formatting
# ============================
def format_nutrition(item):
    name = clean_text(item["name"])

    prompt = random.choice(
        nutrition_prompts
    ).format(food=name)

    serving = item.get("serving_size", "100g")

    lines = [
        f"Calories: {item['calories']} kcal",
        f"Protein: {item['protein']} g",
        f"Carbs: {item['carbs']} g",
        f"Fat: {item['fat']} g",
    ]

    if item.get("fiber") is not None:
        lines.append(f"Fiber: {item['fiber']} g")

    if item.get("sugar") is not None:
        lines.append(f"Sugar: {item['sugar']} g")

    return {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a nutrition assistant. "
                    "Provide short factual nutrition information."
                )
            },
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": (
                    f"{name} nutrition per {serving}:\n"
                    + "\n".join(lines)
                )
            }
        ]
    }


# ============================
# Calories
# ============================
def format_calorie_query(item):
    name = clean_text(item["name"])

    prompt = random.choice(
        calorie_prompts
    ).format(food=name)

    return {
        "messages": [
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": (
                    f"{name} has "
                    f"{item['calories']} kcal "
                    f"per {item.get('serving_size', '100g')}."
                )
            }
        ]
    }


# ============================
# Protein
# ============================
def format_protein_query(item):
    name = clean_text(item["name"])

    prompt = random.choice(
        protein_prompts
    ).format(food=name)

    return {
        "messages": [
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": (
                    f"{name} contains "
                    f"{item['protein']} g protein "
                    f"per {item.get('serving_size', '100g')}."
                )
            }
        ]
    }


# ============================
# Carbs  <-- NEW: was missing
# ============================
def format_carb_query(item):
    name = clean_text(item["name"])

    prompt = random.choice(
        carb_prompts
    ).format(food=name)

    return {
        "messages": [
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": (
                    f"{name} contains "
                    f"{item['carbs']} g carbs "
                    f"per {item.get('serving_size', '100g')}."
                )
            }
        ]
    }


# ============================
# Fat  <-- NEW: was missing
# ============================
def format_fat_query(item):
    name = clean_text(item["name"])

    prompt = random.choice(
        fat_prompts
    ).format(food=name)

    return {
        "messages": [
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": (
                    f"{name} contains "
                    f"{item['fat']} g fat "
                    f"per {item.get('serving_size', '100g')}."
                )
            }
        ]
    }


# ============================
# Macros
# ============================
def format_macro_query(item):
    name = clean_text(item["name"])

    prompt = random.choice(
        macro_prompts
    ).format(food=name)

    return {
        "messages": [
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": (
                    f"{name} macros per "
                    f"{item.get('serving_size', '100g')}:\n"
                    f"Protein: {item['protein']} g\n"
                    f"Carbs: {item['carbs']} g\n"
                    f"Fat: {item['fat']} g\n"
                    f"Calories: {item['calories']} kcal"
                )
            }
        ]
    }


# ============================
# Comparison
# ============================
def format_compare_query(item1, item2):
    name1 = clean_text(item1["name"])
    name2 = clean_text(item2["name"])

    prompt = random.choice(
        compare_prompts
    ).format(
        food1=name1,
        food2=name2
    )

    return {
        "messages": [
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": (
                    f"{name1} vs {name2} "
                    f"(per 100g):\n\n"
                    f"{name1}: "
                    f"{item1['calories']} kcal, "
                    f"{item1['protein']} g protein\n\n"
                    f"{name2}: "
                    f"{item2['calories']} kcal, "
                    f"{item2['protein']} g protein"
                )
            }
        ]
    }


print("Formatting functions loaded successfully.")
nutrition_dataset = nutrition_items

Formatting functions loaded successfully.


## Cell 8: Build the Full Dataset

In [ ]:
# ============================================
# BUILD FINAL TRAINING DATASET
# ============================================

all_data = []

skipped_recipes = 0
skipped_nutrition = 0

# ============================================
# PROCESS NUTRITION ITEMS
# ============================================

print("\nProcessing nutrition data...")

valid_nutrition = []

for item in nutrition_dataset:

    if is_valid_nutrition(item):
        valid_nutrition.append(item)

skipped_nutrition = (
    len(nutrition_dataset) - len(valid_nutrition)
)

print(
    f"Valid nutrition items: "
    f"{len(valid_nutrition)} "
    f"(skipped {skipped_nutrition})"
)

# limit nutrition dataset size
MAX_NUTRITION = 50000

valid_nutrition = valid_nutrition[:MAX_NUTRITION]

for item in valid_nutrition:

    examples = [

        format_nutrition(item),

        format_calorie_query(item),

        format_protein_query(item),

        format_carb_query(item),

        format_fat_query(item),

    ]

    for ex in examples:
        if ex is not None:
            all_data.append(ex)

print(
    f"Nutrition examples generated: "
    f"{len(all_data)}"
)

# ============================================
# PROCESS RECIPES
# ============================================

print("\nProcessing recipes...")

MAX_RECIPES = 50000

recipe_indices = list(range(len(recipe_dataset)))

random.shuffle(recipe_indices)

recipe_count = 0

for idx in recipe_indices:

    if recipe_count >= MAX_RECIPES:
        break

    raw = recipe_dataset[idx]

    if is_valid_recipe(raw):

        formatted = format_recipe(raw)

        if formatted is not None:

            all_data.append(formatted)

            recipe_count += 1

    else:
        skipped_recipes += 1

print(
    f"Recipe examples added: "
    f"{recipe_count} "
    f"(skipped {skipped_recipes})"
)

# ============================================
# ADD COMPARISON EXAMPLES
# ============================================

print("\nGenerating comparison examples...")

NUM_COMPARISONS = min(
    5000,
    len(valid_nutrition) // 2
)

for _ in range(NUM_COMPARISONS):

    item1, item2 = random.sample(
        valid_nutrition,
        2
    )

    compare_example = format_compare_query(
        item1,
        item2
    )

    if compare_example is not None:
        all_data.append(compare_example)

print(
    f"Comparison examples added: "
    f"{NUM_COMPARISONS}"
)

# ============================================
# FILTER BAD EXAMPLES
# ============================================

print("\nFiltering invalid examples...")

clean_data = []

for item in all_data:

    if item is None:
        continue

    if "messages" not in item:
        continue

    # FIX: accept both 2-message (user+assistant) and 3-message (system+user+assistant)
    if len(item["messages"]) not in (2, 3):
        continue

    # user message is last-but-one, assistant is last
    user_msg = item["messages"][-2]["content"]
    assistant_msg = item["messages"][-1]["content"]

    if len(user_msg.strip()) < 3:
        continue

    if len(assistant_msg.strip()) < 5:
        continue

    clean_data.append(item)

all_data = clean_data

# ============================================
# SHUFFLE DATASET
# ============================================

random.shuffle(all_data)

# ============================================
# FINAL STATS
# ============================================

print("\n================================")
print(f"TOTAL TRAINING EXAMPLES: {len(all_data)}")
print("================================")

# ============================================
# SHOW SAMPLE OUTPUTS
# ============================================

print("\nSample examples:\n")

for i in range(min(5, len(all_data))):

    print(f"\n========== Example {i+1} ==========\n")

    print(
        json.dumps(
            all_data[i],
            indent=2,
            ensure_ascii=False
        )
    )

# ============================================
# SAVE DATASET
# ============================================

OUTPUT_FILE = "food_nutrition_dataset.jsonl"

print(f"\nSaving dataset to {OUTPUT_FILE} ...")

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    for item in all_data:

        json.dump(
            item,
            f,
            ensure_ascii=False
        )

        f.write("\n")

print("\nDataset saved successfully.")
print(f"Final dataset size: {len(all_data)} examples")


Processing nutrition data...
Valid nutrition items: 20 (skipped 0)
Nutrition examples generated: 100

Processing recipes...
Recipe examples added: 49983 (skipped 16)

Generating comparison examples...
Comparison examples added: 10

Filtering invalid examples...

TOTAL TRAINING EXAMPLES: 50093

Sample examples:


========== Example 1 ==========

{
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful recipe assistant. Give concise and accurate recipes."
    },
    {
      "role": "user",
      "content": "What is the recipe for Quick And Easy Chicken Cacciatore?"
    },
    {
      "role": "assistant",
      "content": "Recipe: Quick And Easy Chicken Cacciatore\n\nIngredients:\n- 3 to 3 1/2 lb. broiler-fryer chicken, cut up\n- 1 medium green pepper, cored and cut into rings\n- 1 medium onion, cut crosswise into thin slices\n- 1 can whole Italian tomatoes (undrained), cut up (14 oz.)\n- 1 (5 1/2 oz.) can tomato paste\n- 1/4 c. red Burgundy wine\n- 1 tsp. Ital

## Cell 9: Split and Save Dataset

In [ ]:
# 95/5 train/eval split
split_idx = int(0.95 * len(all_data))
train_data = all_data[:split_idx]
eval_data = all_data[split_idx:]

# Save to disk
def save_jsonl(data, filepath):
    with open(filepath, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')

save_jsonl(train_data, '/content/train.jsonl')
save_jsonl(eval_data, '/content/eval.jsonl')

# Also save to Drive as backup
save_jsonl(train_data, f'{PROJECT_DIR}/train.jsonl')
save_jsonl(eval_data, f'{PROJECT_DIR}/eval.jsonl')

print(f"Train: {len(train_data)} examples")
print(f"Eval:  {len(eval_data)} examples")
print(f"\nSaved to /content/ and {PROJECT_DIR}")

# Estimate training time
est_hours = len(train_data) / 100000 * 3.5  # rough: ~3.5h per 100K on T4
print(f"\nEstimated training time on T4: ~{est_hours:.1f} hours")

Train: 47588 examples
Eval:  2505 examples

Saved to /content/ and /content/drive/MyDrive/smollm2-recipe

Estimated training time on T4: ~1.7 hours


## Cell 10: Load Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Set padding token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"Model size: ~{model.num_parameters() * 2 / 1e9:.2f} GB (fp16)")

Loading HuggingFaceTB/SmolLM2-360M...


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded. Parameters: 361,821,120
Model size: ~0.72 GB (fp16)


## Cell 11: Quick Test BEFORE Fine-Tuning
See what the base model outputs so you can compare after training.

In [ ]:
def test_model(model, tokenizer, prompts, max_tokens=150):
    """Test model with a list of prompts."""
    model.eval()
    for prompt in prompts:
        full_prompt = f"User: {prompt}\nAssistant:"
        inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(output[0], skip_special_tokens=True)
        # Only show the assistant's response
        response = response.split("Assistant:")[-1].strip()
        print(f"Q: {prompt}")
        print(f"A: {response[:300]}")
        print("-" * 60)

test_prompts = [
    "How many calories in a banana?",
    "How much protein in chicken breast?",
    "What are the macros for eggs?",
    "Give me the recipe for pancakes",
    "Compare rice and pasta nutritionally",
]

print("===== BASE MODEL (before fine-tuning) =====")
print("(Expect random/bad outputs — this is normal)\n")
test_model(model, tokenizer, test_prompts)

===== BASE MODEL (before fine-tuning) =====
(Expect random/bad outputs — this is normal)

Q: How many calories in a banana?
A: 100 calories.

The following is a list of the most common foods that are high in calories.

Fruit: 100 calories

Cereal: 100 calories

Milk: 100 calories

Candy: 100 calories

Ice cream: 100 calories

Cheese: 100 calories

Bread: 100 calories

Milk: 100 calories

Cereal: 100 calories

Soda: 100 calo
------------------------------------------------------------
Q: How much protein in chicken breast?
A: I'll give you a link to a website that has a lot of protein information.

M
------------------------------------------------------------
Q: What are the macros for eggs?
A: I don't
------------------------------------------------------------
Q: Give me the recipe for pancakes
A: I don't know, I'll have to look
------------------------------------------------------------
Q: Compare rice and pasta nutritionally
A: Compare rice and pasta nutritionally

Compare rice an

## Cell 12: Fine-Tune with LoRA

In [ ]:
!pip install -q trl peft accelerate
!pip install -q --upgrade torchao

import os
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset

# Load dataset
dataset = load_dataset("json", data_files={
    "train": "/content/train.jsonl",
    "eval": "/content/eval.jsonl",
})

print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['eval'])}")

# Convert messages to plain text
def formatting_func(example):
    text = ""

    for msg in example["messages"]:
        role = msg["role"]
        content = msg["content"]

        if role == "system":
            text += f"### System:\n{content}\n\n"
        elif role == "user":
            text += f"### User:\n{content}\n\n"
        elif role == "assistant":
            text += f"### Assistant:\n{content}\n"

    return {"text": text.strip()}

train_dataset = dataset["train"].map(
    formatting_func,
    remove_columns=dataset["train"].column_names
)

eval_dataset = dataset["eval"].map(
    formatting_func,
    remove_columns=dataset["eval"].column_names
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

# Training arguments
training_args = TrainingArguments(
    output_dir=f"{PROJECT_DIR}/checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    fp16=True,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

# Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)

print("\nStarting fine-tuning...")
trainer.model.print_trainable_parameters()

# Resume only if checkpoint exists
checkpoint_dir = f"{PROJECT_DIR}/checkpoints"
has_checkpoint = (
    os.path.exists(checkpoint_dir)
    and any("checkpoint" in x for x in os.listdir(checkpoint_dir))
)

trainer.train(resume_from_checkpoint=has_checkpoint)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.0 MB/s eta 0:00:00


Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Train: 47588 | Eval: 2505


Map:   0%|          | 0/47588 [00:00<?, ? examples/s]

Map:   0%|          | 0/2505 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/47588 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/47588 [00:00<?, ? examples/s]

KeyboardInterrupt: 

## Cell 13: Merge LoRA Weights and Save

In [ ]:
# Merge LoRA adapters back into base model
print("Merging LoRA weights...")
merged_model = trainer.model.merge_and_unload()

# Save locally
LOCAL_MERGED = "/content/smollm2-recipe-merged"
merged_model.save_pretrained(LOCAL_MERGED)
tokenizer.save_pretrained(LOCAL_MERGED)

# Save to Google Drive
DRIVE_MERGED = f"{PROJECT_DIR}/smollm2-recipe-merged"
merged_model.save_pretrained(DRIVE_MERGED)
tokenizer.save_pretrained(DRIVE_MERGED)

print(f"Merged model saved to:")
print(f"  Local:  {LOCAL_MERGED}")
print(f"  Drive:  {DRIVE_MERGED}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_DIR = "/content/drive/MyDrive/RecipeProject/"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

merged_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# ============================================
# EVALUATION METRICS
# ============================================

import math
import re
import numpy as np
import torch

# Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
)

# ============================================
# 1. PERPLEXITY (from eval loss)
# ============================================

eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print("=" * 60)
print("  EVALUATION METRICS")
print("=" * 60)
print(f"\nEval Loss:   {eval_loss:.4f}")
print(f"Perplexity:  {perplexity:.2f}")
print("(Lower is better. Under 10 = good, under 5 = great)\n")


# ============================================
# 2. NUTRITION ACCURACY
# ============================================

# Ground truth for known foods (per 100g)
ground_truth = {
    "banana":          {"calories": 89,  "protein": 1.1,  "carbs": 22.8, "fat": 0.3},
    "chicken breast":  {"calories": 165, "protein": 31.0, "carbs": 0.0,  "fat": 3.6},
    "white rice":      {"calories": 130, "protein": 2.7,  "carbs": 28.2, "fat": 0.3},
    "egg":             {"calories": 155, "protein": 12.6, "carbs": 1.1,  "fat": 10.6},
    "avocado":         {"calories": 160, "protein": 2.0,  "carbs": 8.5,  "fat": 14.7},
    "salmon":          {"calories": 208, "protein": 20.4, "carbs": 0.0,  "fat": 13.4},
    "almonds":         {"calories": 579, "protein": 21.2, "carbs": 21.6, "fat": 49.9},
    "oats":            {"calories": 389, "protein": 16.9, "carbs": 66.3, "fat": 6.9},
    "sweet potato":    {"calories": 90,  "protein": 2.0,  "carbs": 20.7, "fat": 0.1},
    "broccoli":        {"calories": 35,  "protein": 2.4,  "carbs": 7.2, "fat": 0.4},
}


def extract_number(text, keyword):
    """Extract a number that appears near a keyword in text."""
    text = text.lower()

    patterns = [
        rf'{keyword}\w*\s*[:=]?\s*(\d+\.?\d*)',
        rf'(\d+\.?\d*)\s*(?:g|kcal|cal|gram|grams)?\s*{keyword}\w*',
        rf'(\d+\.?\d*)\s*(?:g|kcal|cal|gram|grams)',
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return float(match.group(1))

    return None


def get_model_response(model, tokenizer, query):
    """Get response from the merged model."""

    # Match the same prompt style used during training
    full_prompt = f"""### User:
{query}

### Assistant:
"""

    inputs = tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    # Keep only the assistant response
    if "### Assistant:" in response:
        response = response.split("### Assistant:")[-1]

    # Stop if model starts generating another user turn
    if "### User:" in response:
        response = response.split("### User:")[0]

    return response.strip()


print("-" * 60)
print("NUTRITION ACCURACY TEST")
print("-" * 60)

calorie_errors = []
protein_errors = []
carb_errors = []
fat_errors = []
exact_matches = 0
total_queries = 0

for food, truth in ground_truth.items():
    response = get_model_response(
        merged_model,
        tokenizer,
        f"What is the nutritional info for {food}?"
    )

    pred_cal = extract_number(response, "calori")
    pred_pro = extract_number(response, "protein")
    pred_carb = extract_number(response, "carb")
    pred_fat = extract_number(response, "fat")

    total_queries += 1

    print(f"\n{food.upper()}:")
    print(f"  Response: {response[:200]}")

    if pred_cal is not None:
        err = abs(pred_cal - truth["calories"])
        pct = (err / truth["calories"] * 100) if truth["calories"] > 0 else 0
        calorie_errors.append(pct)

        if pct < 10:
            exact_matches += 1

        print(f"  Calories: predicted={pred_cal}, actual={truth['calories']}, error={pct:.1f}%")
    else:
        print("  Calories: could not parse from response")

    if pred_pro is not None:
        err_pct = (abs(pred_pro - truth["protein"]) / max(truth["protein"], 0.1)) * 100
        protein_errors.append(err_pct)
        print(f"  Protein:  predicted={pred_pro}, actual={truth['protein']}, error={err_pct:.1f}%")

    if pred_carb is not None:
        err_pct = (abs(pred_carb - truth["carbs"]) / max(truth["carbs"], 0.1)) * 100
        carb_errors.append(err_pct)
        print(f"  Carbs:    predicted={pred_carb}, actual={truth['carbs']}, error={err_pct:.1f}%")

    if pred_fat is not None:
        err_pct = (abs(pred_fat - truth["fat"]) / max(truth["fat"], 0.1)) * 100
        fat_errors.append(err_pct)
        print(f"  Fat:      predicted={pred_fat}, actual={truth['fat']}, error={err_pct:.1f}%")

print("\n" + "=" * 60)
print("NUTRITION ACCURACY SUMMARY")
print("=" * 60)
print(f"Foods tested:        {total_queries}")
print(f"Calorie within 10%:  {exact_matches}/{total_queries}")

if calorie_errors:
    print(f"Avg calorie error:   {np.mean(calorie_errors):.1f}%")
if protein_errors:
    print(f"Avg protein error:   {np.mean(protein_errors):.1f}%")
if carb_errors:
    print(f"Avg carb error:      {np.mean(carb_errors):.1f}%")
if fat_errors:
    print(f"Avg fat error:       {np.mean(fat_errors):.1f}%")


# ============================================
# 3. RESPONSE FORMAT SCORE
# ============================================

print("\n" + "-" * 60)
print("RESPONSE FORMAT SCORE")
print("-" * 60)

format_tests = {
    "nutrition": [
        "How many calories in a banana?",
        "What is the nutritional info for salmon?",
        "Protein in chicken breast",
        "What are the macros for Greek yogurt?",
        "How much fat in avocado?",
    ],
    "recipe": [
        "Give me the recipe for pancakes",
        "How do I make fried rice?",
        "Recipe for pasta carbonara",
        "How do you cook scrambled eggs?",
        "Simple way to cook chicken stir fry",
    ],
}

nutrition_format_pass = 0
nutrition_total = len(format_tests["nutrition"])
recipe_format_pass = 0
recipe_total = len(format_tests["recipe"])

# Check nutrition responses have numbers + units
for query in format_tests["nutrition"]:
    response = get_model_response(merged_model, tokenizer, query)

    has_number = bool(re.search(r'\d+\.?\d*', response))
    has_unit = bool(re.search(r'(kcal|cal|g\b|gram|grams)', response.lower()))

    passed = has_number and has_unit

    if passed:
        nutrition_format_pass += 1

    print(f"  {'PASS' if passed else 'FAIL'} | {query}")
    print(f"         {response[:150]}")

print()

# Check recipe responses have ingredients + steps structure
for query in format_tests["recipe"]:
    response = get_model_response(merged_model, tokenizer, query)

    has_ingredients = bool(re.search(r'(ingredient|ingredients|-)', response.lower()))
    has_steps = bool(re.search(r'(step|steps|\d+\.)', response.lower()))

    passed = has_ingredients and has_steps

    if passed:
        recipe_format_pass += 1

    print(f"  {'PASS' if passed else 'FAIL'} | {query}")
    print(f"         {response[:150]}")

print(f"\nNutrition format: {nutrition_format_pass}/{nutrition_total} passed")
print(f"Recipe format:    {recipe_format_pass}/{recipe_total} passed")


# ============================================
# 4. FINAL SCORECARD
# ============================================

overall_format = (
    (nutrition_format_pass + recipe_format_pass)
    / (nutrition_total + recipe_total)
)

print("\n" + "=" * 60)
print("  FINAL SCORECARD")
print("=" * 60)
print(f"  Perplexity:          {perplexity:.2f}")

if calorie_errors:
    print(f"  Calorie accuracy:    {100 - np.mean(calorie_errors):.1f}%")
else:
    print("  Calorie accuracy:    N/A")

print(f"  Format correctness:  {overall_format * 100:.0f}%")
print("  Overall grade:       ", end="")

if perplexity < 5 and overall_format > 0.8:
    print("A — Great!")
elif perplexity < 10 and overall_format > 0.6:
    print("B — Good")
elif perplexity < 20 and overall_format > 0.4:
    print("C — Okay, might need more data or epochs")
else:
    print("D — Needs work, check training loss curve")

print("=" * 60)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


  EVALUATION METRICS

Eval Loss:   2.0730
Perplexity:  7.95
(Lower is better. Under 10 = good, under 5 = great)

------------------------------------------------------------
NUTRITION ACCURACY TEST
------------------------------------------------------------

BANANA:
  Response: Banana has 100 calories, 0 grams fat, 0 grams protein, 0 grams carbohydrates, 0 grams sugar, 0 grams fiber, 0 grams sodium, 0 grams cholesterol, 0 grams potassium, 0 grams calcium, 0 grams iron, 0 gra
  Calories: predicted=100.0, actual=89, error=12.4%
  Protein:  predicted=0.0, actual=1.1, error=100.0%
  Carbs:    predicted=0.0, actual=22.8, error=100.0%
  Fat:      predicted=0.0, actual=0.3, error=100.0%

CHICKEN BREAST:
  Response: Chicken breast has 10 calories per gram.
  Calories: predicted=10.0, actual=165, error=93.9%
  Protein:  predicted=10.0, actual=31.0, error=67.7%
  Carbs:    predicted=10.0, actual=0.0, error=10000.0%
  Fat:      predicted=10.0, actual=3.6, error=177.8%

WHITE RICE:
  Response: Wh

In [ ]:
# ============================================
# 5. ROUGE & BERTScore EVALUATION
# ============================================

!pip install -q rouge-score bert-score

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
import numpy as np

# ============================================
# Build reference/prediction pairs
# ============================================

eval_pairs = {
    "nutrition": [
        {
            "query": "How many calories in a banana?",
            "reference": "Banana has 89 kcal per 100g."
        },
        {
            "query": "How much protein in chicken breast?",
            "reference": "Chicken Breast (cooked) contains 31.0 g protein per 100g."
        },
        {
            "query": "How many carbs in white rice?",
            "reference": "White Rice (cooked) contains 28.2 g carbs per 100g."
        },
        {
            "query": "How much fat in avocado?",
            "reference": "Avocado contains 14.7 g fat per 100g."
        },
        {
            "query": "What is the nutritional info for salmon?",
            "reference": (
                "Salmon (cooked) nutrition per 100g:\n"
                "Calories: 208 kcal\n"
                "Protein: 20.4 g\n"
                "Carbs: 0.0 g\n"
                "Fat: 13.4 g"
            )
        },
        {
            "query": "What are the macros for Greek yogurt?",
            "reference": (
                "Greek Yogurt (plain) macros per 100g:\n"
                "Protein: 10.0 g\n"
                "Carbs: 3.6 g\n"
                "Fat: 0.7 g\n"
                "Calories: 59 kcal"
            )
        },
        {
            "query": "Calories in oats",
            "reference": "Oats (dry) has 389 kcal per 100g."
        },
        {
            "query": "Protein in eggs",
            "reference": "Egg (whole, boiled) contains 12.6 g protein per 100g."
        },
    ],
    "recipe": [
        {
            "query": "Give me the recipe for pancakes",
            "reference": (
                "Recipe: Pancakes\n\n"
                "Ingredients:\n"
                "- 1 cup flour\n"
                "- 1 egg\n"
                "- 1 cup milk\n"
                "- 2 tbsp sugar\n"
                "- 1 tsp baking powder\n"
                "- pinch of salt\n"
                "- butter for cooking\n\n"
                "Steps:\n"
                "1. Mix flour, sugar, baking powder and salt\n"
                "2. Whisk egg and milk together\n"
                "3. Combine wet and dry ingredients\n"
                "4. Heat butter in a pan over medium heat\n"
                "5. Pour batter and cook until bubbles form\n"
                "6. Flip and cook until golden brown"
            )
        },
        {
            "query": "How do I make fried rice?",
            "reference": (
                "Recipe: Fried Rice\n\n"
                "Ingredients:\n"
                "- 2 cups cooked rice (day old)\n"
                "- 2 eggs\n"
                "- 2 tbsp soy sauce\n"
                "- 1 cup mixed vegetables\n"
                "- 2 cloves garlic\n"
                "- sesame oil\n\n"
                "Steps:\n"
                "1. Heat oil in a wok over high heat\n"
                "2. Scramble eggs and set aside\n"
                "3. Stir fry garlic and vegetables\n"
                "4. Add rice and soy sauce\n"
                "5. Mix in scrambled eggs\n"
                "6. Serve hot"
            )
        },
        {
            "query": "Recipe for scrambled eggs",
            "reference": (
                "Recipe: Scrambled Eggs\n\n"
                "Ingredients:\n"
                "- 3 eggs\n"
                "- 1 tbsp butter\n"
                "- salt and pepper\n"
                "- splash of milk\n\n"
                "Steps:\n"
                "1. Crack eggs into a bowl and whisk with milk\n"
                "2. Melt butter in a pan over low heat\n"
                "3. Pour in eggs and stir gently\n"
                "4. Cook until just set, still slightly soft\n"
                "5. Season with salt and pepper"
            )
        },
    ],
}


# ============================================
# Generate predictions
# ============================================

print("Generating model responses...\n")

all_references = []
all_predictions = []
categories = []

for category, pairs in eval_pairs.items():
    for pair in pairs:
        response = get_model_response(
            merged_model, tokenizer, pair["query"]
        )
        all_references.append(pair["reference"])
        all_predictions.append(response)
        categories.append(category)
        print(f"[{category}] {pair['query']}")
        print(f"  Pred: {response[:150]}")
        print()


# ============================================
# ROUGE Scores
# ============================================

print("=" * 60)
print("  ROUGE SCORES")
print("=" * 60)

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge_results = {"nutrition": [], "recipe": []}

for ref, pred, cat in zip(all_references, all_predictions, categories):
    scores = scorer.score(ref, pred)
    rouge_results[cat].append({
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    })

for cat in ["nutrition", "recipe"]:
    results = rouge_results[cat]
    if not results:
        continue
    r1 = np.mean([r["rouge1"] for r in results])
    r2 = np.mean([r["rouge2"] for r in results])
    rL = np.mean([r["rougeL"] for r in results])
    print(f"\n{cat.upper()} ({len(results)} samples):")
    print(f"  ROUGE-1: {r1:.3f}")
    print(f"  ROUGE-2: {r2:.3f}")
    print(f"  ROUGE-L: {rL:.3f}")

# Overall
all_r1 = np.mean([s["rouge1"] for v in rouge_results.values() for s in v])
all_r2 = np.mean([s["rouge2"] for v in rouge_results.values() for s in v])
all_rL = np.mean([s["rougeL"] for v in rouge_results.values() for s in v])

print(f"\nOVERALL ({len(all_references)} samples):")
print(f"  ROUGE-1: {all_r1:.3f}  (word overlap — above 0.4 = good)")
print(f"  ROUGE-2: {all_r2:.3f}  (bigram overlap — above 0.2 = good)")
print(f"  ROUGE-L: {all_rL:.3f}  (longest sequence — above 0.3 = good)")


# ============================================
# BERTScore
# ============================================

print("\n" + "=" * 60)
print("  BERTScore")
print("=" * 60)
print("(Computing — this takes a minute...)\n")

P, R, F1 = bert_score_fn(
    all_predictions,
    all_references,
    lang="en",
    verbose=False,
    device=merged_model.device,
)

bert_results = {"nutrition": [], "recipe": []}

for i, cat in enumerate(categories):
    bert_results[cat].append({
        "precision": P[i].item(),
        "recall": R[i].item(),
        "f1": F1[i].item(),
    })

for cat in ["nutrition", "recipe"]:
    results = bert_results[cat]
    if not results:
        continue
    avg_p = np.mean([r["precision"] for r in results])
    avg_r = np.mean([r["recall"] for r in results])
    avg_f1 = np.mean([r["f1"] for r in results])
    print(f"{cat.upper()} ({len(results)} samples):")
    print(f"  Precision: {avg_p:.3f}")
    print(f"  Recall:    {avg_r:.3f}")
    print(f"  F1:        {avg_f1:.3f}")
    print()

overall_p = P.mean().item()
overall_r = R.mean().item()
overall_f1 = F1.mean().item()

print(f"OVERALL ({len(all_references)} samples):")
print(f"  Precision: {overall_p:.3f}")
print(f"  Recall:    {overall_r:.3f}")
print(f"  F1:        {overall_f1:.3f}")
print(f"  (Above 0.85 = good, above 0.90 = great)")


# ============================================
# COMBINED SUMMARY
# ============================================

print("\n" + "=" * 60)
print("  COMBINED EVALUATION SUMMARY")
print("=" * 60)
print(f"  Perplexity:       {perplexity:.2f}")
print(f"  ROUGE-1:          {all_r1:.3f}")
print(f"  ROUGE-2:          {all_r2:.3f}")
print(f"  ROUGE-L:          {all_rL:.3f}")
print(f"  BERTScore F1:     {overall_f1:.3f}")
if calorie_errors:
    print(f"  Calorie accuracy: {100 - np.mean(calorie_errors):.1f}%")
print(f"  Format score:     {overall_format * 100:.0f}%")
print("=" * 60)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
Generating model responses...

[nutrition] How many calories in a banana?
  Pred: A banana contains 100 calories.

[nutrition] How much protein in chicken breast?
  Pred: Chicken breast contains 10.5 g protein per 100 g.

[nutrition] How many carbs in white rice?
  Pred: White rice has 100 calories per serving.

[nutrition] How much fat in avocado?
  Pred: Avocado contains 10.5 g fat per 100 g.

[nutrition] What is the nutritional info for salmon?
  Pred: Salmon has 10.5 calories per 100g. 10.5 calories per 100g is 10.5 calories per 100g.

[nutrition] What are the macros for Greek yogurt?
  Pred: Greek yogurt has 10 grams of protein per serving.

[nutrition] Calories in oats
  Pred: Oats contain 100 calories per 100g.

[nutrition] Protein in eggs
  Pred: Protein in eggs is 1.5 g/100g.

[recipe] Give me the recipe for pancakes
  Pred: Recipe: Pancakes

Ingredients:
- 1 

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NUTRITION (8 samples):
  Precision: 0.942
  Recall:    0.907
  F1:        0.924

RECIPE (3 samples):
  Precision: 0.871
  Recall:    0.861
  F1:        0.865

OVERALL (11 samples):
  Precision: 0.923
  Recall:    0.895
  F1:        0.908
  (Above 0.85 = good, above 0.90 = great)

  COMBINED EVALUATION SUMMARY
  Perplexity:       7.95
  ROUGE-1:          0.381
  ROUGE-2:          0.141
  ROUGE-L:          0.341
  BERTScore F1:     0.908
  Calorie accuracy: 38.4%
  Format score:     60%


## Cell 14: Test Fine-Tuned Model (Before Quantization)

In [ ]:
print("===== FINE-TUNED MODEL (before quantization) =====")
print("(Should be MUCH better than base model)\n")

test_prompts_v2 = [
    # Nutrition facts
    "How many calories are in 100g of banana?",
    "How much protein is in 100g of chicken breast?",
    "How many carbs are in 100g of white rice?",
    "How much fat is in 100g of avocado?",
    "What is the nutritional information for salmon?",
    "What is the nutritional information for oats?",
    "What is the nutritional information for almonds?",
    "How much protein is in eggs?",

    # Recipe generation
    "Give me a recipe for pancakes.",
    "Give me a recipe for fried rice.",
    "Give me a recipe for scrambled eggs.",
    "Give me a recipe for chicken stir fry."
]

test_model(merged_model, tokenizer, test_prompts_v2)

===== FINE-TUNED MODEL (before quantization) =====
(Should be MUCH better than base model)

Q: How many calories are in 100g of banana?
A: 100g of watermelon contains 200 calories.
------------------------------------------------------------
Q: How much protein is in 100g of chicken breast?
A: 25g

Chicken Breast:
------------------------------------------------------------
Q: How many carbs are in 100g of white rice?
A: 100g of brown rice contains 18g of carbs.

Rice: How many carbs are in 100g of white
------------------------------------------------------------
Q: How much fat is in 100g of avocado?
A: Walnuts are a good source of fat. They are high in monounsaturated fat, which is good
------------------------------------------------------------
Q: What is the nutritional information for salmon?
A: Salmon is a good source of protein, vitamin D, vitamin B12, niacin, vitamin B6, vitamin B3, vitamin B2, vitamin B1, vitamin B5, vitamin B2, vitamin B3, vitamin B1, vitamin B2, vitamin B1

## Cell 15: Convert to GGUF and Quantize

In [ ]:
%%bash

echo "=== Cloning llama.cpp ==="
cd /content
git clone --depth 1 https://github.com/ggerganov/llama.cpp

echo "=== Installing Python requirements ==="
cd /content/llama.cpp
pip install -q -r requirements.txt 2>/dev/null || pip install -q gguf numpy sentencepiece

echo "=== Building llama.cpp (CPU — needed for quantize tool) ==="
cd /content/llama.cpp
make -j$(nproc) llama-quantize 2>/dev/null

echo "=== Done ==="

=== Cloning llama.cpp ===
=== Installing Python requirements ===
=== Building llama.cpp (CPU — needed for quantize tool) ===
=== Done ===


fatal: destination path 'llama.cpp' already exists and is not an empty directory.


In [ ]:
!ls /content/llama.cpp

AGENTS.md	      convert_hf_to_gguf.py	     models
app		      convert_hf_to_gguf_update.py   mypy.ini
AUTHORS		      convert_llama_ggml_to_gguf.py  pocs
benches		      convert_lora_to_gguf.py	     pyproject.toml
build		      docs			     pyrightconfig.json
build-xcframework.sh  examples			     README.md
ci		      flake.nix			     requirements
CLAUDE.md	      ggml			     requirements.txt
cmake		      gguf-py			     scripts
CMakeLists.txt	      grammars			     SECURITY.md
CMakePresets.json     include			     src
CODEOWNERS	      LICENSE			     tests
common		      licenses			     tools
CONTRIBUTING.md       Makefile			     ty.toml
conversion	      media			     vendor


In [ ]:
!find /content/llama.cpp/build -type f | grep -i quant

/content/llama.cpp/build/tools/quantize/cmake_install.cmake
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/DependInfo.cmake
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/compiler_depend.ts
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/cmake_clean.cmake
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/link.txt
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/flags.make
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/progress.make
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/compiler_depend.make
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/depend.make
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize-impl.dir/build.make
/content/llama.cpp/build/tools/quantize/CMakeFiles/progress.marks
/content/llama.cpp/build/tools/quantize/CMakeFiles/llama-quantize.d

In [ ]:
%%bash
cd /content/llama.cpp

cmake --build build --target llama-quantize -j1

[  0%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  0%] Linking CXX static library libcpp-httplib.a
[  0%] Built target cpp-httplib
[  4%] Built target ggml-base
[ 11%] Built target ggml-cpu
[ 13%] Built target ggml
[ 13%] Building CXX object src/CMakeFiles/llama.dir/llama.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-adapter.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-arch.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-batch.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-chat.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-context.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-grammar.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-graph.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-impl.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llama.dir/llama-kv-cache.cpp.o
[ 15%] Building CXX object src/CMakeFiles/llam

In [ ]:
!ls -lh /content/drive/MyDrive/RecipeProject/

total 694M
-rw------- 1 root root  775 May 28 22:33 config.json
-rw------- 1 root root  141 May 28 22:33 generation_config.json
-rw------- 1 root root 691M May 28 22:34 model.safetensors
-rw------- 1 root root  727 May 28 22:33 tokenizer_config.json
-rw------- 1 root root 3.4M May 28 22:33 tokenizer.json


In [ ]:
!ls -lh /content/llama.cpp/build/bin

total 12M
lrwxrwxrwx 1 root root   17 May 28 23:32 libggml-base.so -> libggml-base.so.0
lrwxrwxrwx 1 root root   22 May 28 23:32 libggml-base.so.0 -> libggml-base.so.0.13.0
-rwxr-xr-x 1 root root 857K May 28 23:32 libggml-base.so.0.13.0
lrwxrwxrwx 1 root root   16 May 28 23:33 libggml-cpu.so -> libggml-cpu.so.0
lrwxrwxrwx 1 root root   21 May 28 23:33 libggml-cpu.so.0 -> libggml-cpu.so.0.13.0
-rwxr-xr-x 1 root root 1.2M May 28 23:33 libggml-cpu.so.0.13.0
lrwxrwxrwx 1 root root   12 May 28 23:33 libggml.so -> libggml.so.0
lrwxrwxrwx 1 root root   17 May 28 23:33 libggml.so.0 -> libggml.so.0.13.0
-rwxr-xr-x 1 root root  55K May 28 23:33 libggml.so.0.13.0
lrwxrwxrwx 1 root root   20 May 28 23:54 libllama-common.so -> libllama-common.so.0
lrwxrwxrwx 1 root root   24 May 28 23:54 libllama-common.so.0 -> libllama-common.so.0.0.1
-rwxr-xr-x 1 root root 5.6M May 28 23:54 libllama-common.so.0.0.1
-rwxr-xr-x 1 root root  95K May 28 23:54 libllama-quantize-impl.so
lrwxrwxrwx 1 root root   13 May 

In [ ]:
%%bash

/content/llama.cpp/build/bin/llama-quantize \
    /content/smollm2-recipe-f16.gguf \
    /content/smollm2-recipe-q4.gguf \
    Q4_K_M

echo ""
echo "=== File sizes ==="
ls -lh /content/smollm2-recipe-f16.gguf
ls -lh /content/smollm2-recipe-q4.gguf


llama_quantize: quantize time = 14792.70 ms
llama_quantize:    total time = 14792.70 ms

=== File sizes ===
-rw-r--r-- 1 root root 692M May 28 23:30 /content/smollm2-recipe-f16.gguf
-rw-r--r-- 1 root root 259M May 28 23:58 /content/smollm2-recipe-q4.gguf


llama_print_build_info: build = 1 (19e92c3)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/smollm2-recipe-f16.gguf' to '/content/smollm2-recipe-q4.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 29 key-value pairs and 290 tensors from /content/smollm2-recipe-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = RecipeProject
llama_model_loader: - kv   3:                         general.size_label str              = 362M
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                     

In [ ]:
!ls -lh /content/*.gguf
!cp /content/smollm2-recipe-q4.gguf \
    /content/drive/MyDrive/RecipeProject/

!cp /content/smollm2-recipe-f16.gguf \
    /content/drive/MyDrive/RecipeProject/

-rw-r--r-- 1 root root 692M May 28 23:30 /content/smollm2-recipe-f16.gguf
-rw-r--r-- 1 root root 259M May 28 23:58 /content/smollm2-recipe-q4.gguf


In [ ]:
!ls -lh /content/drive/MyDrive/RecipeProject

total 1.7G
-rw------- 1 root root  775 May 28 22:33 config.json
-rw------- 1 root root  141 May 28 22:33 generation_config.json
-rw------- 1 root root 691M May 28 22:34 model.safetensors
-rw------- 1 root root 692M May 29 00:00 smollm2-recipe-f16.gguf
-rw------- 1 root root 259M May 29 00:00 smollm2-recipe-q4.gguf
-rw------- 1 root root  727 May 28 22:33 tokenizer_config.json
-rw------- 1 root root 3.4M May 28 22:33 tokenizer.json


In [ ]:
import os
import shutil

dst = f"/content/drive/MyDrive/RecipeProject/smollm2-recipe-q4.gguf"

shutil.copy("/content/smollm2-recipe-q4.gguf", dst)

print(f"Quantized model saved to: {dst}")

size_mb = os.path.getsize("/content/smollm2-recipe-q4.gguf") / 1e6
print(f"Model size: {size_mb:.1f} MB")

print("\nThis will fit easily on your 2GB board!")

Quantized model saved to: /content/drive/MyDrive/RecipeProject/smollm2-recipe-q4.gguf
Model size: 270.6 MB

This will fit easily on your 2GB board!


## Cell 16: Test Quantized GGUF Model

In [ ]:
!pip install -q llama-cpp-python
from llama_cpp import Llama

# ============================
# Load quantized GGUF model
# ============================

GGUF_PATH = "/content/drive/MyDrive/RecipeProject/smollm2-recipe-q4.gguf"

print("Loading quantized GGUF model...")

llm = Llama(
    model_path=GGUF_PATH,
    n_ctx=512,
    n_gpu_layers=-1,   # use GPU if available
    verbose=False,
)

print("Model loaded!\n")


# ============================
# Helper function
# ============================

def test_gguf(llm, query, temperature=0.3, max_tokens=200):
    """Test a single query on the GGUF model."""

    prompt = f"""### User:
{query}

### Assistant:
"""

    output = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        repeat_penalty=1.15,
        stop=["### User:", "User:", "\n\n\n"],
        echo=False,
    )

    return output["choices"][0]["text"].strip()


# ============================
# Full test suite
# ============================

print("=" * 60)
print("  QUANTIZED GGUF MODEL TEST RESULTS")
print("=" * 60)

test_cases = [
    # Nutrition facts
    ("How many calories in a banana?", 0.3),
    ("Calories in white rice", 0.3),
    ("How much protein in chicken breast?", 0.3),
    ("Protein in eggs", 0.3),
    ("How many carbs in sweet potato?", 0.3),
    ("Carbs in oats", 0.3),
    ("How much fat in avocado?", 0.3),
    ("Fat in almonds", 0.3),

    # Full nutrition
    ("What is the nutritional info for salmon?", 0.3),
    ("What are the macros for Greek yogurt?", 0.3),

    # Recipes
    ("Give me the recipe for pancakes.", 0.7),
    ("Give me the recipe for fried rice.", 0.7),
    ("Give me the recipe for scrambled eggs.", 0.7),
]

for query, temp in test_cases:
    response = test_gguf(llm, query, temperature=temp)

    print(f"\nQ: {query}")
    print(f"A: {response}")
    print("-" * 60)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
Loading quantized GGUF model...


llama_context: n_ctx_seq (512) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


Model loaded!

  QUANTIZED GGUF MODEL TEST RESULTS

Q: How many calories in a banana?
A: A medium banana contains 104 calories.
------------------------------------------------------------

Q: Calories in white rice
A: White rice has 260 calories per serving.
------------------------------------------------------------

Q: How much protein in chicken breast?
A: Chicken breast contains 25.3 g protein per serving (cooked).
------------------------------------------------------------

Q: Protein in eggs
A: Eggs are a good source of protein.
------------------------------------------------------------

Q: How many carbs in sweet potato?
A: Sweet potato has 10.5 g of carbs per serving (4 oz).
------------------------------------------------------------

Q: Carbs in oats
A: Oats contain 12 grams of carbohydrates per serving.
------------------------------------------------------------

Q: How much fat in avocado?
A: Avocado contains 13.5 g fat, of which 0.8% is saturated fat and 46.9 mg chol

## Cell 17: Interactive Chat Mode
Try your own queries!

In [ ]:
print("===== Interactive Recipe & Nutrition Bot =====")
print("Type your question and press Enter.")
print("Type 'quit' to exit.\n")

while True:
    query = input("You: ").strip()
    if query.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break
    if not query:
        continue

    # Use lower temperature for nutrition, higher for recipes
    is_recipe = any(word in query.lower() for word in ['recipe', 'make', 'cook', 'prepare'])
    temp = 0.7 if is_recipe else 0.3

    response = test_gguf(llm, query, temperature=temp, max_tokens=300)
    print(f"Bot: {response}\n")

===== Interactive Recipe & Nutrition Bot =====
Type your question and press Enter.
Type 'quit' to exit.

You: Give me the recipe for pancakes
Bot: Recipe: Pancakes

Ingredients:
- 1/2 c. shortening, melted (not butter)
- 3 Tbsp. milk
- 4 eggs
- 1 tsp. baking powder
- pinch of salt
- dash of pepper
- 6 to 8 slices sugarless pancake mix or regular flour and a little water

Steps:
1. Mix ingredients well, except for the powdered butter (if using)

You: I want the recipe for kimchi
Bot: Recipe: Kimchi

Ingredients:
- 1 medium head cabbage, shredded (about 6 c.)
- 2 red bell peppers, chopped
- salt to taste
- 3 or 4 scallions, sliced fine
- 1 large onion, cut up small pieces
- chili powder
- 2 Tbsp. oil
- pepper and garlic sauce for seasoning

Steps:
1. Combine cabbage with other ingredients; mix well.
2. Cover tightly in jar (or glass) or plastic bag to keep from drying out.



## Cell 18: Download Model for Your Board

In [ ]:
# Option 1: Download directly from Colab
from google.colab import files
files.download('/content/smollm2-recipe-q4.gguf')

# Option 2: It's already on Google Drive at:
print(f"\nAlso available on Google Drive: {PROJECT_DIR}/smollm2-recipe-q4.gguf")

## Cell 19: Board Deployment Instructions

Run these commands on your 2GB board:

```bash
# 1. Build llama.cpp on the board
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
make -j$(nproc)

# 2. Copy your model to the board
# (via USB, scp, etc)

# 3. Run inference
./llama-cli \
    -m /path/to/smollm2-recipe-q4.gguf \
    -p "User: How much protein in chicken?\nAssistant:" \
    -n 128 \
    --temp 0.3 \
    -t 4

# 4. For a simple server
./llama-server \
    -m /path/to/smollm2-recipe-q4.gguf \
    --host 0.0.0.0 \
    --port 8080 \
    -t 4
```

Then you can send requests:
```bash
curl http://localhost:8080/completion \
    -d '{"prompt": "User: Calories in rice?\nAssistant:", "n_predict": 64, "temperature": 0.3}'
```

---
## Troubleshooting

| Issue | Fix |
|---|---|
| Colab disconnects during training | Reduce MAX_RECIPES to 20K, or use Colab Pro |
| CUDA out of memory | Reduce per_device_train_batch_size to 2 |
| USDA API rate limited | Use DEMO_KEY sparingly, or get a free key from fdc.nal.usda.gov |
| Model gives gibberish | Check training loss — should decrease. Try more epochs |
| convert_hf_to_gguf fails | Update llama.cpp: `cd llama.cpp && git pull` |
| Model too big for board | Use Q4_0 instead of Q4_K_M (slightly worse quality, smaller) |